# 01 EDA comercial - Mini Trade Promo Optimizer

Tercera iteracion del EDA. Evalua calidad, soporte historico y relaciones comerciales relevantes para uplift y ROI. Los resultados son descriptivos: no identifican efectos causales ni entrenan modelos.

## 1. Objetivo y contexto de negocio

**Pregunta analitica:** que evidencia historica permite aprender la respuesta promocional y limitar una futura recomendacion por SKU y cadena?

**Implicacion de negocio:** descuento y duracion son palancas; uplift mide respuesta porcentual y ROI el retorno oficial observado. Deben evaluarse conjuntamente con volumen base, elasticidad y soporte local.

## 2. Trazabilidad del codigo

La logica se centraliza en modulos reutilizables para evitar inconsistencias, permitir pruebas unitarias y facilitar la ejecucion del pipeline fuera de Jupyter. El notebook explica y visualiza los outputs; no replica sus implementaciones.

In [ ]:
import pandas as pd

traceability = pd.DataFrame([
    ["Preparacion completa", "run_preparation", "mini_tpo.pipeline", "data/ y reports/", "Una ruta reproducible de generacion"],
    ["Soporte SKU x cadena", "build_sku_chain_support", "mini_tpo.support_analysis", "support_sku_chain.csv", "Controlar extrapolacion local"],
    ["Reglas de soporte", "support_rules_table", "mini_tpo.support_analysis", "vista del notebook", "Hacer visible la heuristica"],
    ["Perfil temporal", "build_temporal_profile", "mini_tpo.support_analysis", "temporal_profile_*.csv", "Diagnosticar cambios de mix"],
    ["Resumen descuento/uplift", "discount_uplift_summary", "mini_tpo.support_analysis", "discount_uplift_summary.csv", "Medir respuesta por palanca"],
    ["Resumen descuento/ROI", "discount_roi_summary", "mini_tpo.support_analysis", "discount_roi_summary.csv", "Mostrar trade-off de retorno"],
    ["Resumen duracion", "duration_*_summary", "mini_tpo.support_analysis", "duration_*_summary.csv", "Comparar respuesta y retorno"],
    ["Interaccion descriptiva", "elasticity_discount_uplift_tables", "mini_tpo.support_analysis", "elasticity_discount_*_matrix.csv", "Revisar heterogeneidad"],
    ["Auditoria derivada", "add_audit_columns", "mini_tpo.data_validation", "clean full", "Verificar consistencia y leakage"],
    ["Manifest", "create_feature_manifest", "mini_tpo.feature_manifest", "feature_manifest.json", "Contrato de variables"],
], columns=["resultado_o_tabla", "funcion_responsable", "modulo", "output_generado", "proposito"])
display(traceability)

## 3. Configuracion y ejecucion reproducible

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "configs" / "project_config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import importlib

from mini_tpo.data_loading import load_config, read_raw_data, read_clean_full, read_modeling_data
import mini_tpo.support_analysis as support_analysis_module
import mini_tpo.pipeline as pipeline_module

# Evita imports obsoletos al reejecutar esta celda en un kernel que ya estaba abierto.
support_analysis_module = importlib.reload(support_analysis_module)
pipeline_module = importlib.reload(pipeline_module)
support_rules_table = support_analysis_module.support_rules_table
run_preparation = pipeline_module.run_preparation

cfg = load_config(ROOT / "configs" / "project_config.yaml")
TABLES_DIR = ROOT / cfg["project"]["tables_dir"]
FIGURES_DIR = ROOT / cfg["project"]["figures_dir"]
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# Unica ejecucion del pipeline en este notebook.
pipeline_result = run_preparation()
raw = read_raw_data(cfg)
clean_full = read_clean_full(cfg)
modeling = read_modeling_data(cfg)

## 4. Fuentes y versiones de datos

El raw sirve para auditar la fuente sin modificaciones. Clean full conserva todo el historico con tipos, missing explicito, flags y variables de auditoria. Modeling restringe el universo al dominio futuro de 5%-40% de descuento y 5-21 dias.

In [ ]:
comparison = pd.DataFrame([
    ["Raw", len(raw), raw.shape[1], "Sin modificaciones"],
    ["Clean full", len(clean_full), clean_full.shape[1], "Tipos, flags, missing explicito y auditoria"],
    ["Modeling domain", len(modeling), modeling.shape[1], "Solo 5%-40% y 5-21 dias"],
], columns=["dataset", "filas", "columnas", "tratamientos"])
display(comparison)

## 5. Calidad del raw

Esta tabla se calcula mediante `validation_summary()` definida en `src/mini_tpo/data_validation.py` y usa exclusivamente el dataset raw.

In [ ]:
validation = pipeline_result["validation_summary"]
display(validation)
quality_raw = pd.DataFrame({
    "dtype_raw": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_pct": raw.isna().mean(),
    "n_unique": raw.nunique(dropna=False),
}).reset_index(names="variable")
display(quality_raw)

**Hallazgo:** la calidad original se conserva visible antes de cualquier tratamiento. **Implicacion tecnica:** los faltantes y registros fuera de dominio se marcan, no se borran silenciosamente. **Implicacion de negocio:** evita confundir ausencia de registro con ausencia real de una mecanica. **Limitacion:** las reglas detectan inconsistencias, pero no sustituyen conocimiento operacional.

## 6. Transformaciones para analisis y cobertura temporal

In [ ]:
coverage = pd.DataFrame({
    "metrica": ["fecha_min", "fecha_max", "skus", "marcas", "cadenas", "sku_x_cadena", "fuera_dominio"],
    "valor": [clean_full.fecha_inicio_tanda.min(), clean_full.fecha_inicio_tanda.max(), clean_full.id_material.nunique(), clean_full.des_marca.nunique(), clean_full.subcadena.nunique(), clean_full[["id_material", "subcadena"]].drop_duplicates().shape[0], clean_full.flag_fuera_dominio_optimizacion.sum()],
})
display(coverage)
monthly = pd.read_csv(TABLES_DIR / "temporal_profile_monthly.csv")
display(monthly)

**Hallazgo:** la cobertura temporal no es equivalente a soporte local. **Implicacion tecnica:** el futuro split debe ser temporal y auditar presencia de SKU/cadena en train y holdout. **Implicacion de negocio:** una recomendacion para un SKU reciente tendra mayor incertidumbre. **Limitacion:** los cambios mensuales tambien reflejan cambios de mix.

## 7. Soporte por SKU x cadena

Esta tabla se calcula mediante `build_sku_chain_support()` definida en `src/mini_tpo/support_analysis.py`. Dataset: clean full.

In [ ]:
support = pd.read_csv(TABLES_DIR / "support_sku_chain.csv")
display(support)
display(support["clasificacion_soporte"].value_counts().rename_axis("nivel").to_frame("combinaciones"))
pivot = support.pivot(index="id_material", columns="subcadena", values="promociones")
fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="YlGnBu")
ax.set_xticks(range(len(pivot.columns)), pivot.columns)
ax.set_yticks(range(len(pivot.index)), pivot.index)
ax.set_title("Promociones por SKU y cadena")
ax.set_xlabel("Cadena"); ax.set_ylabel("SKU")
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, int(pivot.iloc[i, j]), ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="Promociones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "support_sku_chain_heatmap.png", dpi=150); plt.show()

**Hallazgo:** el soporte es heterogeneo. **Implicacion tecnica:** el optimizador futuro debe limitar recomendaciones por evidencia local, no solo por rango global. **Implicacion de negocio:** una combinacion permitida puede seguir siendo riesgosa para un SKU y cadena concretos. **Limitacion:** cantidad y diversidad son proxies de soporte, no garantias de precision.

## 8. Reglas visibles de soporte

Esta tabla se calcula mediante `support_rules_table()` con umbrales leidos desde `configs/project_config.yaml`.

In [ ]:
support_rules = support_rules_table(cfg["business_rules"]["support_thresholds"]).rename(columns={
    "nivel": "Nivel",
    "min_observations": "Numero minimo de observaciones",
    "min_discount_bands": "Cobertura de descuento",
    "min_durations": "Cobertura de duracion",
})
display(support_rules)

> Los niveles son una heuristica de control de riesgo para esta prueba tecnica, no una regla universal de negocio.

## 9. Espacio descuento x duracion

In [ ]:
discount_support = pd.read_csv(TABLES_DIR / "support_discount_by_sku_chain.csv")
duration_support = pd.read_csv(TABLES_DIR / "support_duration_by_sku_chain.csv")
display(discount_support.head(20)); display(duration_support.head(20))
domain_counts = modeling.groupby(["factor_descuento", "duracion_dias"], observed=True).size().unstack(fill_value=0)
display(domain_counts)

**Hallazgo:** no todas las combinaciones tienen el mismo soporte. **Implicacion tecnica:** la optimizacion debe restringirse a celdas observadas o declarar extrapolacion. **Impacto de negocio:** reduce recomendaciones con ROI incierto. **Limitacion:** el soporte agregado puede ocultar vacios por SKU y cadena.

## 10. Descuento versus uplift

Esta tabla se calcula mediante `discount_uplift_summary()` en `mini_tpo.support_analysis`. Dataset: modeling domain.

In [ ]:
discount_uplift = pd.read_csv(TABLES_DIR / "discount_uplift_summary.csv")
display(discount_uplift)
fig, ax = plt.subplots(figsize=(9, 4))
for chain, group in discount_uplift.groupby("subcadena"):
    ax.plot(group["banda_descuento_opt"], group["uplift_mediano"], marker="o", label=chain)
ax.set_title("Uplift mediano por banda de descuento y cadena")
ax.set_xlabel("Banda de descuento"); ax.set_ylabel("Uplift mediano"); ax.legend(title="Cadena")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "discount_vs_uplift.png", dpi=150); plt.show()

**Pregunta:** como cambia la respuesta con una palanca controlable? **Hallazgo:** el uplift tiende a aumentar con el descuento, con dispersion y diferencias por cadena. **Implicacion tecnica:** descuento solo no explica la respuesta; se requieren SKU, cadena, baseline y elasticidad. **Implicacion de negocio:** mas profundidad puede elevar volumen, pero no garantiza rentabilidad. **Limitacion:** asociacion historica, no efecto causal.

## 11. Descuento versus ROI

Esta tabla se calcula mediante `discount_roi_summary()` en `mini_tpo.support_analysis`. Los extremos permanecen en la tabla; solo se limita el eje del boxplot.

In [ ]:
discount_roi = pd.read_csv(TABLES_DIR / "discount_roi_summary.csv")
display(discount_roi)
bands = [x for x in modeling.assign(banda=pd.cut(modeling.factor_descuento, bins=[.05,.10,.15,.20,.25,.30,.35,.4000001], include_lowest=True, right=False)).banda.dropna().cat.categories]
box_data = [modeling.loc[pd.cut(modeling.factor_descuento, bins=[.05,.10,.15,.20,.25,.30,.35,.4000001], include_lowest=True, right=False).eq(b), "roi"] for b in bands]
lo, hi = modeling.roi.quantile([0.01, 0.99])
fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(box_data, tick_labels=[str(b) for b in bands], showfliers=True)
ax.set_ylim(lo, hi); ax.set_title("ROI por banda de descuento (eje P1-P99)")
ax.set_xlabel("Banda de descuento"); ax.set_ylabel("ROI oficial"); ax.tick_params(axis="x", rotation=35)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "discount_vs_roi.png", dpi=150); plt.show()

**Hallazgo:** media, mediana y porcentaje negativo no evolucionan igual que uplift. **Implicacion tecnica:** uplift y ROI requieren targets separados o una regla de decision conjunta. **Implicacion de negocio:** descuentos mayores pueden aumentar volumen, pero tambien elevar inversion promocional y reducir retorno. **Limitacion:** la formula interna del KPI no se descompone y no se atribuye causalidad.

## 12. Duracion versus uplift

Esta tabla se calcula mediante `duration_uplift_summary()` en `mini_tpo.support_analysis`. Dataset: modeling domain.

In [ ]:
duration_uplift = pd.read_csv(TABLES_DIR / "duration_uplift_summary.csv")
display(duration_uplift)
fig, ax = plt.subplots(figsize=(8, 4))
for chain, group in duration_uplift.groupby("subcadena"):
    ax.plot(group["duracion_dias"], group["uplift_mediano"], marker="o", label=chain)
ax.set_title("Uplift mediano por duracion y cadena"); ax.set_xlabel("Duracion (dias)"); ax.set_ylabel("Uplift mediano"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "duration_vs_uplift.png", dpi=150); plt.show()

**Hallazgo:** una tanda mas larga no implica necesariamente mayor uplift porcentual. **Implicacion tecnica:** duracion debe modelarse como palanca y posible interaccion. **Implicacion de negocio:** aun con uplift similar, mas dias aumentan volumen base acumulado y exposicion financiera. **Limitacion:** las duraciones son discretas y su mix no es aleatorio.

## 13. Duracion versus ROI

Esta tabla se calcula mediante `duration_roi_summary()` en `mini_tpo.support_analysis`.

In [ ]:
duration_roi = pd.read_csv(TABLES_DIR / "duration_roi_summary.csv")
display(duration_roi)
fig, ax = plt.subplots(figsize=(8, 4))
aggregate_duration_roi = duration_roi.groupby("duracion_dias").agg(roi_mediano=("roi_mediano", "median"), pct_roi_negativo=("pct_roi_negativo", "mean"), observaciones=("observaciones", "sum")).reset_index()
ax.plot(aggregate_duration_roi.duracion_dias, aggregate_duration_roi.roi_mediano, marker="o", label="ROI mediano")
ax.set_title("ROI mediano por duracion"); ax.set_xlabel("Duracion (dias)"); ax.set_ylabel("ROI oficial")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "duration_vs_roi.png", dpi=150); plt.show()

**Hallazgo:** retorno y tasa negativa cambian entre duraciones, junto con el tamano de muestra. **Implicacion tecnica:** la validacion debera ponderar incertidumbre por soporte. **Implicacion de negocio:** extender una promocion puede acumular volumen e inversion sin mejorar el ratio. **Limitacion:** no se afirma que la duracion cause el ROI observado.

## 14. Elasticidad versus uplift

In [ ]:
rho, pvalue = spearmanr(modeling["elasticidad_estimada"], modeling["uplift_real"], nan_policy="omit")
elasticity_spearman = pd.DataFrame([{"spearman_rho": rho, "p_value": pvalue, "observaciones": len(modeling)}])
display(elasticity_spearman)
fig, ax = plt.subplots(figsize=(8, 5))
hb = ax.hexbin(modeling.elasticidad_estimada, modeling.uplift_real, gridsize=30, mincnt=1, cmap="viridis")
ax.set_title("Elasticidad estimada versus uplift")
ax.set_xlabel("Elasticidad estimada"); ax.set_ylabel("Uplift real")
fig.colorbar(hb, ax=ax, label="Observaciones")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "elasticity_vs_uplift.png", dpi=150); plt.show()

La elasticidad representa el cambio porcentual de cantidad ante un cambio porcentual de precio. **Hallazgo:** existe heterogeneidad aun para valores cercanos de elasticidad. **Implicacion tecnica:** en feature engineering se evaluara `factor_descuento * abs(elasticidad_estimada)`. **Implicacion de negocio:** la misma profundidad puede producir respuestas distintas segun sensibilidad historica. **Limitacion:** es una estimacion interna, no evidencia causal.

## 15. Volumen base y tamano del impacto

Esta tabla se calcula mediante `baseline_impact_summary()` usando variables de auditoria creadas por `add_audit_columns()`. Dataset: modeling domain.

In [ ]:
baseline_impact = pd.read_csv(TABLES_DIR / "baseline_impact_summary.csv")
display(baseline_impact)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(modeling.volumen_base_sem, modeling.uplift_real, alpha=.3, s=12)
axes[0].set(xlabel="Volumen base semanal", ylabel="Uplift", title="Baseline versus uplift")
axes[1].scatter(modeling.volumen_base_sem, modeling.roi, alpha=.3, s=12)
axes[1].set(xlabel="Volumen base semanal", ylabel="ROI", title="Baseline versus ROI")
axes[2].scatter(modeling.volumen_base_sem, modeling.audit_volumen_incremental_observado, alpha=.3, s=12)
axes[2].set(xlabel="Volumen base semanal", ylabel="Volumen incremental", title="Baseline versus impacto absoluto")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "baseline_vs_impact.png", dpi=150); plt.show()

`volumen_base_tanda = volumen_base_sem * duracion_dias / 7`; el volumen incremental observado es volumen promocional menos ese baseline. **Hallazgo:** un mismo uplift representa unidades distintas segun escala. **Implicacion tecnica:** el modelo final debe evaluar error porcentual y error en unidades. **Implicacion de negocio:** 30% sobre 100 unidades no equivale a 30% sobre 5,000. **Limitacion:** las variables incrementales son auditoria postpromocion, nunca features.

## 16. Promocion secundaria

Esta tabla se calcula mediante `secondary_promo_summary()` en `mini_tpo.support_analysis`.

In [ ]:
secondary = pd.read_csv(TABLES_DIR / "secondary_promo_summary.csv")
display(secondary)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].bar(secondary.flag_secundario, secondary.uplift_mediano, color=["#4E79A7", "#F28E2B", "#59A14F"])
axes[0].set(title="Uplift mediano", xlabel="Promocion secundaria", ylabel="Uplift")
axes[1].bar(secondary.flag_secundario, secondary.roi_mediano, color=["#4E79A7", "#F28E2B", "#59A14F"])
axes[1].set(title="ROI mediano", xlabel="Promocion secundaria", ylabel="ROI")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "secondary_promo_summary.png", dpi=150); plt.show()

**Hallazgo:** secundaria, no secundaria y desconocido difieren tambien en descuento y duracion. **Implicacion tecnica:** el flag funciona como control de mecanica y el missing no se imputa a cero. **Implicacion de negocio:** otra activacion simultanea puede alterar la respuesta observada. **Limitacion:** las diferencias mezclan seleccion comercial y composicion; no son efecto causal.

## 17. Interaccion descriptiva descuento x elasticidad

Estas matrices se calculan mediante `elasticity_discount_uplift_tables()`; uplift y conteos siempre se muestran juntos.

In [ ]:
interaction_uplift = pd.read_csv(TABLES_DIR / "elasticity_discount_uplift_matrix.csv")
interaction_counts = pd.read_csv(TABLES_DIR / "elasticity_discount_count_matrix.csv")
display(interaction_uplift); display(interaction_counts)
u = interaction_uplift.set_index("banda_elasticidad")
n = interaction_counts.set_index("banda_elasticidad")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, matrix, title, cmap in [(axes[0], u, "Uplift mediano", "YlGnBu"), (axes[1], n, "Conteo", "Greys")]:
    im = ax.imshow(matrix.values.astype(float), aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(matrix.columns)), matrix.columns, rotation=35, ha="right")
    ax.set_yticks(range(len(matrix.index)), matrix.index)
    ax.set_title(title); ax.set_xlabel("Banda de descuento"); ax.set_ylabel("Banda de elasticidad")
    fig.colorbar(im, ax=ax)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "elasticity_discount_heatmaps.png", dpi=150); plt.show()

**Hallazgo:** la respuesta mediana varia entre celdas y algunas tienen soporte limitado. **Implicacion tecnica:** la interaccion es candidata futura, no una feature base ya construida. **Implicacion de negocio:** permite distinguir descuentos semejantes aplicados a sensibilidades distintas. **Limitacion:** no interpretar celdas pequenas sin revisar conteos.

## 18. Uplift y posible piso

Esta tabla se calcula mediante `floor_summary()` en `mini_tpo.support_analysis`.

In [ ]:
floor_sku = pd.read_csv(TABLES_DIR / "uplift_floor_by_sku.csv")
floor_time = pd.read_csv(TABLES_DIR / "uplift_floor_by_time.csv")
display(floor_sku); display(floor_time)
print(f"Registros en piso: {clean_full.flag_uplift_en_piso.sum()} ({clean_full.flag_uplift_en_piso.mean():.2%})")

**Hallazgo:** existe concentracion exacta cerca de 0.05. **Implicacion tecnica:** revisar residuos y calibracion alrededor del piso. **Implicacion de negocio:** puede alterar recomendaciones de baja respuesta. **Limitacion:** puede ser censura, redondeo, regla interna o comportamiento genuino; no se corrige sin evidencia.

## 19. ROI oficial y valores extremos

Estas tablas se calculan mediante `roi_distribution_audit()` y `roi_tail_tables()`; no se winsoriza el KPI.

In [ ]:
display(pd.read_csv(TABLES_DIR / "roi_distribution_audit.csv"))
display(pd.read_csv(TABLES_DIR / "top_10_roi.csv"))
display(pd.read_csv(TABLES_DIR / "bottom_10_roi.csv"))

Para esta prueba se asume que `roi` es el KPI oficial calculado por Alicorp y se acepta como target valido. Como su formula interna no esta disponible, no se realiza una descomposicion contable ni se atribuyen efectos causales a sus componentes. Las colas se conservan para model risk y validacion robusta.

## 20. Evolucion temporal

Esta tabla se calcula mediante `build_temporal_profile()` en `mini_tpo.support_analysis`.

In [ ]:
display(monthly)
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(monthly.periodo, monthly.uplift_mediano, marker="o"); axes[0].set(title="Uplift mediano mensual", ylabel="Uplift")
axes[1].plot(monthly.periodo, monthly.roi_mediano, marker="o", color="#F28E2B"); axes[1].set(title="ROI mediano mensual", xlabel="Mes", ylabel="ROI")
axes[1].tick_params(axis="x", rotation=60)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "temporal_uplift_roi.png", dpi=150); plt.show()

**Hallazgo:** resultados y politica promocional varian en el tiempo. **Implicacion tecnica:** el futuro holdout debe respetar cronologia y medir drift. **Implicacion de negocio:** una regla aprendida en un mix anterior puede perder vigencia. **Limitacion:** el perfil no separa tiempo, SKU, cadena y estrategia.

## 21. Cold start

Esta tabla se calcula mediante `sku_history_profile()` en `mini_tpo.support_analysis`.

In [ ]:
sku_history = pd.read_csv(TABLES_DIR / "sku_history_profile.csv")
display(sku_history)
display(sku_history.clasificacion_sku.value_counts().to_frame("skus"))

**Hallazgo:** hay SKUs recientes frente a otros maduros. **Implicacion tecnica:** conviene comparar modelos globales que compartan informacion; marca puede evaluarse para cold start. **Implicacion de negocio:** evita promesas de precision sin historia suficiente. **Limitacion:** la taxonomia es heuristica.

## 22. Leakage y disponibilidad

Esta tabla se calcula mediante `feature_availability_audit()` en `mini_tpo.data_audit`; las reconciliaciones derivadas provienen de `add_audit_columns()`.

In [ ]:
availability = pd.read_csv(TABLES_DIR / "feature_availability_audit.csv")
display(availability)
audit_checks = clean_full[["audit_diff_abs_uplift_real", "audit_diff_rel_venta_promo", "audit_diff_rel_inversion_promo"]].describe()
display(audit_checks)

`volumen_base_sem` y `elasticidad_estimada` son prepromocion validas por supuesto de la prueba. `volumen_promo`, ventas, inversion, uplift y ROI son resultados posteriores. Una correlacion alta no vuelve valida una variable si no existe al decidir; por eso ROI es target, nunca predictor.

## 23. Implicaciones para el futuro modelado

In [ ]:
manifest = json.loads((ROOT / cfg["project"]["feature_manifest"]).read_text(encoding="utf-8"))
manifest_view = pd.DataFrame(manifest["variables"])
display(manifest_view.loc[manifest_view.name.isin(manifest["features_model_base"] + ["roi", "precio_base", "des_marca"])])

El baseline usará SKU, cadena, descuento, duración, volumen base, elasticidad y promoción secundaria, porque están disponibles antes de la tanda y describen su contexto y condiciones. La fecha se reservará para el orden y el split temporal. Precio y marca se evaluarán por sensibilidad, pues son redundantes con el SKU. Los targets, variables postpromoción y de auditoría se excluyen para evitar fuga de información. La evaluación incluirá error porcentual y error en unidades para reflejar tanto la precisión estadística como el impacto comercial.

## 24. Resumen ejecutivo business driven

In [ ]:
business_summary = pd.DataFrame([
    ["Uplift aumenta con profundidad en el agregado", "Medianas por banda con dispersion y diferencias por cadena", "Modelar no linealidad e interacciones", "Mas volumen potencial"],
    ["ROI no replica la trayectoria del uplift", "Mediana y porcentaje negativo por banda", "Targets separados y decision conjunta", "Evitar destruir valor con descuentos altos"],
    ["Respuesta heterogenea por SKU y cadena", "Soporte y resultados locales distintos", "Modelo global con controles locales", "Recomendaciones adaptadas"],
    ["Volumen base escala el impacto", "Uplift similar produce unidades incrementales distintas", "Evaluar error en porcentaje y unidades", "Priorizar impacto absoluto"],
    ["Elasticidad diferencia sensibilidad", "Uplift varia por bandas de elasticidad y descuento", "Evaluar interaccion en la siguiente fase", "Mejor discriminacion promocional"],
    ["Flag secundario controla otra mecanica", "Grupos difieren en uplift, ROI y diseno", "Mantener categoria desconocido", "Evitar atribuir respuesta solo al descuento"],
    ["Maximizar uplift no basta", "Retorno, inversion y soporte imponen trade-offs", "Optimizacion multiobjetivo restringida", "Balancear crecimiento y rentabilidad"],
], columns=["hallazgo", "evidencia", "decision_futura", "impacto_de_negocio"])
display(business_summary)
business_summary.to_csv(TABLES_DIR / "eda_business_driven_summary.csv", index=False)

## 25. Limitaciones

- Analisis observacional y descriptivo; no prueba causalidad.
- ROI se acepta como KPI oficial, pero no se descompone su formula interna.
- Los umbrales de soporte y cold start son heuristicas de control de riesgo.
- `row_id` es estable para el raw actual mientras se preserve su orden.
- No se entreno modelo, no se fijo split definitivo y no se construyo optimizador.